In [134]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv('../data/clean_online_retail.csv', parse_dates=['InvoiceDate'])
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [135]:
df = df[~((df['CustomerID'] == 16446) & (df['StockCode'] == '23843'))]
df = df[~((df['CustomerID'] == 12346) & (df['StockCode'] == '23166'))]

In [136]:
best_sellers = df.groupby('StockCode')['Quantity'].sum().sort_values(ascending=False).reset_index()
best_sellers

,StockCode,Quantity
0,84077,54319
1,22197,49160
2,85099B,46078
3,85123A,36763
4,84879,35263
...,...,...
3656,23630,1
3657,22218,1
3658,23664,1
3659,84227,1


In [137]:
non_product_codes = ['C2', 'BANK CHARGES']  # PADS مستبعدة لحد ما نتأكد من وصفها
df = df[~df['StockCode'].isin(non_product_codes)]

In [138]:
df[df['StockCode'] == 'PADS']['Description'].unique()

array(['PADS TO MATCH ALL CUSHIONS'], dtype=object)

In [139]:
suspicious_codes = ['C2', 'BANK CHARGES', 'PADS', 'D', 'CRUK', 'DCGS0076', 'AMAZONFEE']
df[df['StockCode'].isin(suspicious_codes)]['StockCode'].value_counts()

StockCode
PADS    4
Name: count, dtype: int64

In [140]:
product_stats = df.groupby('StockCode').agg(
    total_quantity=('Quantity', 'sum'),
    unique_customers=('CustomerID', 'nunique')
).reset_index()
product_stats['avg_qty_per_customer'] = product_stats['total_quantity'] / product_stats['unique_customers']
product_stats.sort_values('total_quantity', ascending=False).head(10)

,StockCode,total_quantity,unique_customers,avg_qty_per_customer
2803,84077,54319,307,176.934853
1088,22197,49160,407,120.786241
3218,85099B,46078,635,72.563780
3232,85123A,36763,856,42.947430
3058,84879,35263,678,52.010324
423,21212,33670,635,53.023622
1919,23084,27153,450,60.340000
1352,22492,26076,213,122.422535
1469,22616,25329,195,129.892308
910,21977,24230,410,59.097561


In [141]:
avg_price=df.groupby('StockCode')['UnitPrice'].mean().reset_index()
avg_price

,StockCode,UnitPrice
0,10002,0.850000
1,10080,0.411905
2,10120,0.210000
3,10123C,0.650000
4,10124A,0.420000
...,...,...
3654,90214V,0.930000
3655,90214W,0.290000
3656,90214Y,0.610000
3657,90214Z,0.290000


In [142]:
# def recommend(avg_price, best_sellers, budget):
#     needed = 10
#     margins = [0.20, 0.25, 0.30, 0.35, 0.40]
#     recommended_codes = set()
#     recommendations = []
#     for margin in margins:
#         if margin == 0.20:
#             lower_bound = budget - (budget * margin)
#             upper_bound = budget + (budget * margin)
#         else:
#             pre_lower_bound = lower_bound
#             lower_bound = budget - (budget * margin)
#             upper_bound = pre_lower_bound
#         budget_products = avg_price[(avg_price['UnitPrice'] >= lower_bound) &(avg_price['UnitPrice'] <= upper_bound)]
#         recommended_items = budget_products.merge(best_sellers,on='StockCode')
#         recommended_items['price_distance'] = abs(recommended_items['UnitPrice'] - budget)
#         recommended_items = recommended_items.sort_values(by=['price_distance', 'Quantity'],ascending=[True, False])
#         new_items = recommended_items[~recommended_items['StockCode'].isin(recommended_codes)]
#         selected_items = new_items[:needed]
#         recommendations.append(selected_items)
#         recommended_codes.update(selected_items['StockCode'])
#         needed -= len(selected_items)
#         if needed == 0:
#             break
#     if needed > 0:
#         fallback = avg_price[(avg_price['UnitPrice'] <= budget) &(~avg_price['StockCode'].isin(recommended_codes))]
#         fallback = fallback.merge(best_sellers,on='StockCode')
#         fallback['price_distance'] = abs(fallback['UnitPrice'] - budget)
#         fallback = fallback.sort_values(by=['price_distance', 'Quantity'],ascending=[True, False])
#         selected_items = fallback[:needed]
#         recommendations.append(selected_items)
#     final_recommendations = pd.concat(recommendations,ignore_index=True)
#     final_recommendations = final_recommendations.sort_values(by=['price_distance'])
#     return final_recommendations.head(10)

In [143]:
def recommend(avg_price, best_sellers, budget, n=10):
    cheaper_or_equal = avg_price[avg_price['UnitPrice'] <= budget].copy()
    more_expensive = avg_price[avg_price['UnitPrice'] > budget].copy()
    cheaper_or_equal = cheaper_or_equal.sort_values('UnitPrice', ascending=False)
    more_expensive = more_expensive.sort_values('UnitPrice', ascending=True)
    combined = pd.concat([cheaper_or_equal, more_expensive], ignore_index=True)

    return combined.head(n)

In [144]:
recommend(avg_price,best_sellers,80)

,StockCode,UnitPrice
0,21769,66.360000
1,22929,63.915385
2,84632,59.950000
3,84963B,49.950000
4,22833,45.635000
5,23064,43.011364
6,84963A,42.950000
7,84816,39.950000
8,21686,39.672222
9,20785,39.150000


In [145]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [146]:
conf_matrix=df.pivot_table(index='CustomerID',columns='StockCode',values='Quantity',aggfunc='sum',fill_value=0)
conf_matrix

StockCode,10002,10080,10120,10123C,10124A,10124G,10125,10133,10135,11001,...,90214P,90214R,90214S,90214T,90214U,90214V,90214W,90214Y,90214Z,PADS
CustomerID,,,,,,,,,,,,,,,,,,,,,
12347,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12348,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12349,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12350,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
12352,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18280,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
18281,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
18282,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [147]:
from scipy.sparse import csr_matrix

customer_product_sparse = csr_matrix(conf_matrix)

In [148]:
print(customer_product_sparse.shape)
print(conf_matrix.memory_usage().sum())  
print(customer_product_sparse.data.nbytes) 

(4334, 3659)
126899520
2129824


In [149]:
from sklearn.metrics.pairwise import cosine_similarity
item_similarity = cosine_similarity(customer_product_sparse.T)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=conf_matrix.columns,
    columns=conf_matrix.columns
)

item_similarity_df.shape

(3659, 3659)

In [150]:
def recommend_from_history(customer_id, conf_matrix, item_similarity_df, n=10):
    purchased_products = conf_matrix.loc[customer_id]
    purchased_products = purchased_products[purchased_products > 0].index    
    similarity_scores = item_similarity_df.loc[purchased_products].sum(axis=0)
    similarity_scores = similarity_scores.drop(purchased_products)
    top_n_recommendations = similarity_scores.sort_values(ascending=False).head(n)
    return top_n_recommendations

In [151]:
result = recommend_from_history(12347, conf_matrix, item_similarity_df)
result.sort_values(ascending=False).head(15)

StockCode
22907    20.128562
23110    20.037985
22966    19.175363
22505    19.090580
22557    18.970447
22629    18.868534
22908    18.778079
22630    18.664626
22757    18.635055
20725    18.618547
dtype: float64

In [152]:
recommend_from_history(12347, conf_matrix, item_similarity_df, n=10)

StockCode
22907    20.128562
23110    20.037985
22966    19.175363
22505    19.090580
22557    18.970447
22629    18.868534
22908    18.778079
22630    18.664626
22757    18.635055
20725    18.618547
dtype: float64

In [153]:
product_description = (
    df.groupby('StockCode')['Description']
      .first()
      .reset_index()
)

product_lookup = avg_price.merge(
    product_description,
    on='StockCode'
)

product_lookup = product_lookup[
    ['StockCode', 'Description', 'UnitPrice']
]

product_lookup.head()

,StockCode,Description,UnitPrice
0,10002,INFLATABLE POLITICAL GLOBE,0.850000
1,10080,GROOVY CACTUS INFLATABLE,0.411905
2,10120,DOGGY RUBBER,0.210000
3,10123C,HEARTS WRAPPING TAPE,0.650000
4,10124A,SPOTS ON RED BOOKCOVER TAPE,0.420000


In [154]:
def unified_recommend(
    is_new_customer,
    product_lookup,
    customer_id=None,
    budget=None,
    conf_matrix=None,
    item_similarity_df=None,
    avg_price=None,
    best_sellers=None,
    n=10
):
    
    # New customer
    if is_new_customer:
        if budget is not None:
            # New customer + budget
            raw_result = recommend(
                avg_price,
                best_sellers,
                budget
            )
        else:
            # New customer without budget
            raw_result = best_sellers.head(n)
        
        # StockCode is a normal column
        product_codes = raw_result['StockCode'].tolist()
    
    # Existing customer
    else:
        raw_result = recommend_from_history(
            customer_id,
            conf_matrix,
            item_similarity_df,
            n
        )
        
        # StockCode is the index
        product_codes = raw_result.index.tolist()
    
    # Convert StockCodes into DataFrame
    codes_df = pd.DataFrame({
        'StockCode': product_codes
    })
    
    # Add Description and UnitPrice
    final_result = codes_df.merge(
        product_lookup,
        on='StockCode',
        how='left'
    )
    return final_result.head(n)

In [155]:
# 1. New customer بدون budget
unified_recommend(
    is_new_customer=True,
    product_lookup=product_lookup,
    best_sellers=best_sellers,
    n=10
)

,StockCode,Description,UnitPrice
0,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,0.292606
1,22197,SMALL POPCORN HOLDER,0.839208
2,85099B,JUMBO BAG RED RETROSPOT,2.015969
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.892768
4,84879,ASSORTED COLOUR BIRD ORNAMENT,1.680710
5,21212,PACK OF 72 RETROSPOT CAKE CASES,0.548181
6,23084,RABBIT NIGHT LIGHT,2.012770
7,22492,MINI PAINT SET VINTAGE,0.656523
8,22616,PACK OF 12 LONDON TISSUES,0.325816
9,21977,PACK OF 60 PINK PAISLEY CAKE CASES,0.551420


In [156]:
# 2. New customer + budget
unified_recommend(
    is_new_customer=True,
    product_lookup=product_lookup,
    avg_price=avg_price,
    best_sellers=best_sellers,
    budget=80,
    n=10
)

,StockCode,Description,UnitPrice
0,21769,VINTAGE POST OFFICE CABINET,66.360000
1,22929,SCHOOL DESK AND CHAIR,63.915385
2,84632,DECORATIVE HANGING SHELVING UNIT,59.950000
3,84963B,BLUE PAINTED KASHMIRI CHAIR,49.950000
4,22833,HALL CABINET WITH 3 DRAWERS,45.635000
5,23064,CINDERELLA CHANDELIER,43.011364
6,84963A,PINK PAINTED KASHMIRI CHAIR,42.950000
7,84816,DANISH ROSE BEDSIDE CABINET,39.950000
8,21686,MEDINA STAMPED METAL STOOL,39.672222
9,20785,FUSCHIA RETRO BAR STOOL,39.150000


In [157]:
# 3. Existing customer
unified_recommend(
    is_new_customer=False,
    product_lookup=product_lookup,
    customer_id=17850,
    conf_matrix=conf_matrix,
    item_similarity_df=item_similarity_df,
    n=10
)

,StockCode,Description,UnitPrice
0,23284,DOORMAT KEEP CALM AND COME IN,7.911963
1,22767,TRIPLE PHOTO FRAME CORNICE,9.849231
2,21428,SET3 BOOK BOX GREEN GINGHAM FLOWER,4.180824
3,22365,DOORMAT RESPECTABLE HOUSE,7.862063
4,22960,JAM MAKING SET WITH JARS,4.178364
5,20685,DOORMAT RED RETROSPOT,7.758094
6,23412,HEART MIRROR ANTIQUE WHITE,7.394583
7,15056N,EDWARDIAN PARASOL NATURAL,5.864237
8,48184,DOORMAT ENGLISH ROSE,7.776011
9,22169,FAMILY ALBUM WHITE PICTURE FRAME,8.388252


In [158]:
import joblib
joblib.dump(item_similarity_df, '../models/item_similarity.pkl')
joblib.dump(product_lookup, '../models/product_lookup.pkl')
joblib.dump(best_sellers, '../models/best_sellers.pkl')

['../models/best_sellers.pkl']

In [159]:
joblib.dump(conf_matrix, '../models/conf_matrix.pkl')

['../models/conf_matrix.pkl']

In [160]:
joblib.dump(avg_price, '../models/avg_price.pkl')

['../models/avg_price.pkl']

In [163]:
recommend(avg_price, best_sellers, 0.01)

,StockCode,UnitPrice
0,PADS,0.000750
1,16045,0.043478
2,16218,0.073200
3,16219,0.090851
4,16216,0.094634
5,17038,0.095263
6,16259,0.096774
7,16161G,0.100000
8,17136A,0.102941
9,20668,0.118478
